# Algorithm Interview Preparation Notebook

This notebook covers **foundational data structures & algorithms** (entry-to-intermediate) followed by **Bloomberg-style advanced problems**.

## Table of Contents

1. **Arrays & Strings** — A Very Big Sum, Left Rotation
2. **Linked Lists** — Insert a Node at a Position, Cycle Detection
3. **Stacks & Queues** — Balanced Brackets, Queue Using Two Stacks
4. **Hash Maps** — Ice Cream Parlour
5. **Sorting Algorithms** — Insertion Sort Part 1 & 2
6. **Trees** — Binary Tree Insertion, Height of a Binary Tree
7. **Graphs (BFS & DFS)** — Breadth First Search, Snakes and Ladders
8. **Recursion** — Fibonacci Numbers
9. **Sliding Window, Swapping & Rolling Algorithms** — sliding window max, reversal, rolling hash
10. **Bloomberg-Style Advanced Problems** — LRU Cache, Top-K, Merge Intervals, Currency Conversion, RandomizedSet

---
## 1. Arrays & Strings

**Topics:** Basic I/O, array manipulation, rotation.

**Problems:**
- A Very Big Sum (entry-level — familiarise yourself with the platform)
- Left Rotation

**Solution descriptions:**
- **A Very Big Sum:** Straightforward O(N) sum — demonstrates basic iteration and large integer handling. Key insight: Python handles big ints natively, so just `sum(arr)` works.
- **Left Rotation:** Array slicing trick `arr[d:] + arr[:d]` gives O(N) time, O(N) space. Alternative: reverse the whole array, then reverse each partition for O(1) extra space — a two-pointer swapping technique that's useful when memory is constrained.

### Arrays & Strings — Tests

In [ ]:
# A Very Big Sum — sum of large integers (simple warm-up)
def a_very_big_sum(arr):
    return sum(arr)

# Left Rotation — shift array elements left by d positions
def left_rotate(arr, d):
    n = len(arr)
    d %= n
    return arr[d:] + arr[:d]


In [ ]:
def test_arrays():
    assert a_very_big_sum([1000000001, 1000000002, 1000000003]) == 3000000006
    assert left_rotate([1, 2, 3, 4, 5], 2) == [3, 4, 5, 1, 2]
    assert left_rotate([1, 2, 3, 4, 5], 7) == [3, 4, 5, 1, 2]  # d > n
    assert left_rotate([1], 0) == [1]
    print("Arrays & Strings: all tests passed")

test_arrays()


---
## 2. Linked Lists

**Topics:** Node insertion, cycle detection (Floyd's algorithm).

**Problems:**
- Insert a Node at a Position Given in a List
- Cycle Detection

**Solution descriptions:**
- **Insert Node at Position:** Traverse to `position - 1`, link new node to `curr.next`, then point `curr.next` to new node. O(N) worst-case. Edge case: position 0 means new head.
- **Cycle Detection (Floyd's Tortoise & Hare):** Two pointers — slow moves 1 step, fast moves 2. If they meet, a cycle exists. O(N) time, O(1) space. Key invariants: fast never skips over slow in a cycle; if fast reaches `None`, no cycle.

In [ ]:
class _ListNode:
    __slots__ = ("data", "next")
    def __init__(self, data=0, next_node=None):
        self.data = data
        self.next = next_node

# Insert a Node at a Position Given in a List
def insert_node_at_position(head, data, position):
    """Insert a new node with `data` at index `position` (0-based)."""
    node = _ListNode(data)
    if position == 0:
        node.next = head
        return node
    curr = head
    for _ in range(position - 1):
        curr = curr.next
    node.next = curr.next
    curr.next = node
    return head

# Cycle Detection — Floyd's Tortoise and Hare
def has_cycle(head):
    """Return True if the linked list has a cycle, False otherwise."""
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False


### Linked Lists — Tests

In [ ]:
def _list_to_array(head):
    result = []
    while head:
        result.append(head.data)
        head = head.next
    return result

def test_linked_lists():
    # Insert at position
    head = None
    for i in range(3):
        head = insert_node_at_position(head, i, i)
    assert _list_to_array(head) == [0, 1, 2]
    head = insert_node_at_position(head, 99, 1)
    assert _list_to_array(head) == [0, 99, 1, 2]

    # Cycle detection
    a, b, c = _ListNode(1), _ListNode(2), _ListNode(3)
    a.next = b; b.next = c
    assert has_cycle(a) == False
    c.next = a  # create cycle
    assert has_cycle(a) == True

    print("Linked Lists: all tests passed")

test_linked_lists()


---
## 3. Stacks & Queues

**Topics:** LIFO/FIFO patterns, amortised analysis.

**Problems:**
- Balanced Brackets
- Queue Using Two Stacks

**Solution descriptions:**
- **Balanced Brackets:** Push opening brackets onto a stack; on a closing bracket, pop and verify it matches. O(N) time, O(N) space. Trap: unmatched closing bracket or leftover stack means imbalance.
- **Queue Using Two Stacks:** Use one stack for enqueue and another for dequeue. When dequeue stack is empty, drain enqueue stack into it, reversing order. Amortised O(1) per operation — each element moves stacks at most twice.

In [ ]:
# Balanced Brackets
def is_balanced(s):
    """Check if brackets are properly matched and nested."""
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in pairs.values():  # opening
            stack.append(ch)
        elif ch in pairs:          # closing
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack


# Queue Using Two Stacks
class QueueTwoStacks:
    """O(1) amortised enqueue/dequeue using two stacks."""

    def __init__(self):
        self._inbox = []   # push here
        self._outbox = []  # pop/peek from here

    def enqueue(self, val):
        self._inbox.append(val)

    def dequeue(self):
        if not self._outbox:
            while self._inbox:
                self._outbox.append(self._inbox.pop())
        return self._outbox.pop()

    def peek(self):
        if not self._outbox:
            while self._inbox:
                self._outbox.append(self._inbox.pop())
        return self._outbox[-1] if self._outbox else None


### Stacks & Queues — Tests

In [ ]:
def test_stacks_queues():
    # Balanced Brackets
    assert is_balanced("{[()]}") == True
    assert is_balanced("{[(])}") == False
    assert is_balanced("{{[[(())]]}}") == True
    assert is_balanced("") == True
    assert is_balanced("[") == False

    # Queue Two Stacks
    q = QueueTwoStacks()
    q.enqueue(1); q.enqueue(2); q.enqueue(3)
    assert q.peek() == 1
    assert q.dequeue() == 1
    assert q.dequeue() == 2
    q.enqueue(4)
    assert q.dequeue() == 3
    assert q.dequeue() == 4
    assert q.peek() is None

    print("Stacks & Queues: all tests passed")

test_stacks_queues()


---
## 4. Hash Maps

**Topics:** Lookup tables, two-sum pattern.

**Problem:**
- Ice Cream Parlour — find two prices that sum to exactly the money available.

**Solution descriptions:**
- **Ice Cream Parlour (Two Sum):** Iterate once. For each price, compute `money - price` and check if that complement is already in a hash map. If yes, return both indices (1-based). O(N) time, O(N) space. Edge cases: duplicate values, no solution.


In [ ]:
# Ice Cream Parlour — two-sum variant
def ice_cream_parlour(money, prices):
    """Return the 1-based indices of two flavours that sum to `money`."""
    seen = {}  # price -> index (1-based)
    for i, p in enumerate(prices, start=1):
        needed = money - p
        if needed in seen:
            return sorted([seen[needed], i])
        seen[p] = i
    return []


### Hash Maps — Tests

In [ ]:
def test_hash_maps():
    assert ice_cream_parlour(4, [1, 4, 5, 3, 2]) == [1, 4]
    assert ice_cream_parlour(4, [2, 2, 3]) == [1, 2]  # duplicate values
    assert ice_cream_parlour(10, [1, 2, 3]) == []      # no solution
    print("Hash Maps: all tests passed")

test_hash_maps()


---
## 5. Sorting Algorithms

**Topics:** Insertion sort mechanics, in-place shifting.

**Problems:**
- Insertion Sort — Part 1 (insert last element into sorted array)
- Insertion Sort — Part 2 (full sort)

**Solution descriptions:**
- **Part 1:** The array is fully sorted except the last element. Walk backwards from `n-2`, shifting elements right until finding the correct spot for `last`. O(N).
- **Part 2:** Classic insertion sort — maintain a sorted prefix; repeatedly take the next element and insert it into the correct position by shifting. O(N²) worst-case, O(1) space. Good for nearly-sorted data.

In [ ]:
# Insertion Sort — Part 1: insert last element into a sorted array
def insertion_sort_one(arr):
    """Move the last element into its correct position in a sorted array."""
    n = len(arr)
    if n <= 1:
        return arr
    last = arr[-1]
    i = n - 2
    while i >= 0 and arr[i] > last:
        arr[i + 1] = arr[i]
        i -= 1
    arr[i + 1] = last
    return arr


# Insertion Sort — Part 2: full sort
def insertion_sort(arr):
    """Sort the array in-place using insertion sort. O(N^2)."""
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr


### Sorting — Tests

In [ ]:
def test_sorting():
    # Part 1
    assert insertion_sort_one([2, 3, 4, 5, 1]) == [1, 2, 3, 4, 5]
    assert insertion_sort_one([1]) == [1]
    assert insertion_sort_one([]) == []

    # Part 2
    assert insertion_sort([4, 3, 2, 1]) == [1, 2, 3, 4]
    assert insertion_sort([1, 2, 3]) == [1, 2, 3]
    assert insertion_sort([]) == []

    print("Sorting: all tests passed")

test_sorting()


---
## 6. Trees

**Topics:** BST properties, recursive tree traversal.

**Problems:**
- Binary Tree Insertion
- Height of a Binary Tree

**Solution descriptions:**
- **Binary Tree Insertion:** Recursively descend left if `data < root.data`, right otherwise. Insert as a leaf. O(log N) average, O(N) worst (skewed tree).
- **Height of a Binary Tree:** Recursively compute `1 + max(height(left), height(right))`. Base case: leaf or `None` returns 0. O(N) time, O(H) recursion stack space.

In [ ]:
class _TreeNode:
    __slots__ = ("data", "left", "right")
    def __init__(self, data=0):
        self.data = data
        self.left = None
        self.right = None

# Binary Tree Insertion
def bst_insert(root, data):
    """Insert `data` into a BST. Returns the root."""
    if root is None:
        return _TreeNode(data)
    if data < root.data:
        root.left = bst_insert(root.left, data)
    else:
        root.right = bst_insert(root.right, data)
    return root


# Height of a Binary Tree
def tree_height(root):
    """Return the height (number of edges on longest path root→leaf)."""
    if root is None or (root.left is None and root.right is None):
        return 0
    return 1 + max(tree_height(root.left), tree_height(root.right))


### Trees — Tests

In [ ]:
def _bst_inorder(root, out=None):
    if out is None:
        out = []
    if root:
        _bst_inorder(root.left, out)
        out.append(root.data)
        _bst_inorder(root.right, out)
    return out

def test_trees():
    root = None
    for v in [4, 2, 6, 1, 3, 5, 7]:
        root = bst_insert(root, v)
    assert _bst_inorder(root) == [1, 2, 3, 4, 5, 6, 7]
    assert tree_height(root) == 2           # 4 → 6 → 7  (2 edges)
    assert tree_height(None) == 0
    assert tree_height(_TreeNode(1)) == 0   # single node
    print("Trees: all tests passed")

test_trees()


---
## 7. Graphs (BFS & DFS)

**Topics:** BFS for shortest paths, graph modelling.

**Problems:**
- Breadth First Search (shortest reach in unweighted graph)
- Snakes and Ladders (minimum moves)

**Solution descriptions:**
- **BFS Shortest Reach:** Standard BFS from start node; each edge weight = 6. Track distances in an array initialised to -1. O(V + E).
- **Snakes and Ladders:** Model the board as a graph of 100 squares. BFS from square 1; each dice roll (1–6) is an edge. If a square has a snake/ladder, teleport immediately. First time reaching square 100 is minimal moves. O(N) where N = board size.

In [ ]:
from collections import defaultdict, deque
from typing import List

# BFS — shortest reach in unweighted graph
def bfs_shortest_reach(n: int, edges: List[List[int]], start: int) -> List[int]:
    """Return distances from `start` to all nodes (1..n). Distance = 6 per edge."""
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    dist = [-1] * (n + 1)
    dist[start] = 0
    q = deque([start])

    while q:
        u = q.popleft()
        for v in graph[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 6
                q.append(v)
    return dist[1:]


# Snakes and Ladders — minimum moves
def snakes_and_ladders(board: List[int]) -> int:
    """Board is 0-indexed length N. board[i] = destination or -1.
    Return min dice rolls (1..6) to reach last square."""
    n = len(board)
    dist = [-1] * n
    dist[0] = 0
    q = deque([0])

    while q:
        u = q.popleft()
        if u == n - 1:
            return dist[u]
        for dice in range(1, 7):
            v = u + dice
            if v >= n:
                continue
            if board[v] != -1:
                v = board[v]
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return -1


### Graphs — Tests

In [ ]:
def test_graphs():
    # BFS shortest reach
    dist = bfs_shortest_reach(5, [(1, 2), (1, 3)], 1)
    assert dist == [0, 6, 6, -1, -1]

    # Snakes and Ladders
    board = [-1] * 30
    board[2] = 21   # ladder: square 3 → 22 (0-indexed: 2 → 21)
    board[26] = 0   # snake:  square 27 → 1  (0-indexed: 26 → 0)
    assert snakes_and_ladders(board) == 3

    print("Graphs: all tests passed")

test_graphs()


---
## 8. Recursion

**Topics:** Recursive thinking, memoisation, iterative DP.

**Problem:**
- Fibonacci Numbers

**Solution descriptions:**
- **Fibonacci Numbers (three approaches):**
  1. **Naive recursion** — `fib(n) = fib(n-1) + fib(n-2)`. O(2^N) — only for tiny n.
  2. **Memoised recursion** — cache computed values in a dict. O(N) time, O(N) space.
  3. **Iterative** — two variables `a, b` updated in a loop. O(N) time, O(1) space — the optimal solution for production.

In [ ]:
# Fibonacci Numbers — recursive and iterative

def fib_recursive(n):
    """O(2^N) — naive recursion. Fine for small n."""
    if n <= 1:
        return n
    return fib_recursive(n - 1) + fib_recursive(n - 2)


def fib_memo(n, memo=None):
    """O(N) — top-down DP with memoisation."""
    if memo is None:
        memo = {0: 0, 1: 1}
    if n not in memo:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]


def fib_iterative(n):
    """O(N) — bottom-up iterative."""
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


### Recursion — Tests

In [ ]:
def test_fib():
    assert fib_recursive(0) == 0
    assert fib_recursive(1) == 1
    assert fib_recursive(10) == 55

    assert fib_memo(0) == 0
    assert fib_memo(10) == 55
    assert fib_memo(50) == 12586269025   # large — memoisation handles it

    assert fib_iterative(0) == 0
    assert fib_iterative(10) == 55
    assert fib_iterative(50) == 12586269025

    print("Recursion: all tests passed")

test_fib()


---
## 1. LRU Cache — O(1) get / put

**Why they ask:** Caches are core to low-latency products like the Bloomberg Terminal (pricing data, news, entity lookups).

**What they look for:** Hash map + doubly-linked list invariants; clean edge-case handling.

**Follow-ups:**
- Add TTL expiration.
- Make it thread-safe.
- How do you test race conditions?

---
## 9. Sliding Window, Swapping & Rolling Algorithms

**Why it matters:** Sliding windows are used everywhere in finance (rolling averages, time windows). Swapping/reversal algorithms build array manipulation fundamentals. Rolling hashes power string matching in text processing.

### 🧠 Beginner's Guide

**Sliding Window** is a technique where you maintain a "window" over a subset of data and slide it across. Instead of recomputing from scratch at each position, you update the window incrementally — O(N) instead of O(N·K).

*Intuition:* Imagine looking through a train window at the passing landscape. The window only shows a small slice at a time. As the train moves, you don't re-examine everything — you just see what enters and leaves your view.

**Key patterns:**
- **Fixed window:** Window of size K slides one step at a time
- **Variable window:** Window expands/contracts based on conditions (e.g. smallest subarray with sum ≥ target)

**Swapping & Reversal:** Reversing arrays or swapping elements in-place is the foundation of more complex algorithms (array rotations, linked list reversals, sorting).

**Rolling Hash (Rabin-Karp):** A hash that can be updated incrementally as the window slides — computing the new hash from the old hash in O(1) instead of O(K).

### Problems Covered

| Problem | Technique | Complexity |
|---|---|---|
| **Sliding Window Maximum** | Deque (monotonic queue) | O(N) |
| **Longest Substring Without Repeating** | Variable sliding window + hash set | O(N) |
| **Array Reversal (in-place)** | Two-pointer swap | O(N), O(1) space |
| **Rolling Hash (Rabin-Karp)** | Polynomial rolling hash | O(N) average |
| **Minimum Size Subarray Sum** | Variable sliding window | O(N) |

In [ ]:
from collections import deque
from typing import List, Optional

# ════════════════════════════════════════════════════════════
# 1. SLIDING WINDOW MAXIMUM (Deque / Monotonic Queue)
# ════════════════════════════════════════════════════════════
def sliding_window_max(nums: List[int], k: int) -> List[int]:
    """Return max of each sliding window of size k. O(N) using a deque.

    The deque stores INDICES of elements in decreasing order of value.
    - When a new element arrives, pop smaller elements from the back
    - The front of the deque is always the max of the current window
    - Remove indices outside the window from the front
    """
    dq = deque()  # stores indices, values decreasing
    result = []

    for i, val in enumerate(nums):
        # Remove smaller elements from back (they'll never be max again)
        while dq and nums[dq[-1]] < val:
            dq.pop()

        dq.append(i)

        # Remove index outside window from front
        if dq[0] <= i - k:
            dq.popleft()

        # Start recording once we have a full window
        if i >= k - 1:
            result.append(nums[dq[0]])

    return result


# ════════════════════════════════════════════════════════════
# 2. LONGEST SUBSTRING WITHOUT REPEATING CHARACTERS
# ════════════════════════════════════════════════════════════
def longest_unique_substring(s: str) -> int:
    """Longest substring with all unique characters. O(N) variable sliding window."""
    seen = {}  # char -> last seen index
    left = 0
    max_len = 0

    for right, ch in enumerate(s):
        if ch in seen and seen[ch] >= left:
            left = seen[ch] + 1  # shrink window past the duplicate
        seen[ch] = right
        max_len = max(max_len, right - left + 1)

    return max_len


# ════════════════════════════════════════════════════════════
# 3. IN-PLACE ARRAY REVERSAL (Two-pointer swap)
# ════════════════════════════════════════════════════════════
def reverse_array(arr: List[int], left: int = 0, right: Optional[int] = None) -> None:
    """Reverse arr[left:right+1] in-place using two-pointer swap. O(N), O(1) space."""
    if right is None:
        right = len(arr) - 1
    while left < right:
        arr[left], arr[right] = arr[right], arr[left]
        left += 1
        right -= 1


# ════════════════════════════════════════════════════════════
# 4. ARRAY ROTATION via REVERSAL (O(1) space alternative to slicing)
# ════════════════════════════════════════════════════════════
def rotate_array_reversal(arr: List[int], k: int) -> None:
    """Rotate array RIGHT by k positions using three reversals. O(1) extra space.

    The trick: reverse the whole array, then reverse each partition.
    Example: [1,2,3,4,5], k=2
      Step 1: reverse all → [5,4,3,2,1]
      Step 2: reverse first k → [4,5,3,2,1]
      Step 3: reverse rest → [4,5,1,2,3] ✓
    """
    n = len(arr)
    if n == 0:
        return
    k %= n
    reverse_array(arr, 0, n - 1)
    reverse_array(arr, 0, k - 1)
    reverse_array(arr, k, n - 1)


# ════════════════════════════════════════════════════════════
# 5. ROLLING HASH (Rabin-Karp algorithm)
# ════════════════════════════════════════════════════════════
def rolling_hash_search(text: str, pattern: str) -> List[int]:
    """Find all occurrences of pattern in text using Rabin-Karp rolling hash.

    Uses a polynomial hash: H(s) = Σ s[i] * base^i  (mod modulus)
    Rolling property: when sliding window, we can compute new hash from old
    in O(1) by subtracting the outgoing character and adding the incoming one.
    """
    if not pattern or len(pattern) > len(text):
        return []

    base = 256       # larger than character set
    mod = 1_000_000_007  # large prime to reduce collisions
    n, m = len(text), len(pattern)

    # Precompute base^(m-1) mod mod
    highest_power = pow(base, m - 1, mod)

    # Compute hash of pattern and first window
    pattern_hash = 0
    window_hash = 0
    for i in range(m):
        pattern_hash = (pattern_hash * base + ord(pattern[i])) % mod
        window_hash = (window_hash * base + ord(text[i])) % mod

    matches = []
    for i in range(n - m + 1):
        if pattern_hash == window_hash:
            # Hash match — verify (spurious collision possible)
            if text[i:i + m] == pattern:
                matches.append(i)

        # Roll the hash forward (unless this is the last window)
        if i < n - m:
            window_hash = (window_hash - ord(text[i]) * highest_power) % mod
            window_hash = (window_hash * base + ord(text[i + m])) % mod

    return matches


# ════════════════════════════════════════════════════════════
# 6. MINIMUM SIZE SUBARRAY SUM (Variable sliding window)
# ════════════════════════════════════════════════════════════
def min_subarray_len(target: int, nums: List[int]) -> int:
    """Smallest subarray length with sum ≥ target. O(N) variable window."""
    left = 0
    total = 0
    min_len = float("inf")

    for right, val in enumerate(nums):
        total += val
        while total >= target:
            min_len = min(min_len, right - left + 1)
            total -= nums[left]
            left += 1

    return 0 if min_len == float("inf") else min_len


### Sliding Window, Swapping & Rolling — Tests

In [ ]:
def test_sliding_swapping_rolling():
    # 1. Sliding Window Maximum
    assert sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
    assert sliding_window_max([1, -1], 1) == [1, -1]
    assert sliding_window_max([1], 1) == [1]
    print("✓ Sliding Window Maximum")

    # 2. Longest Unique Substring
    assert longest_unique_substring("abcabcbb") == 3  # "abc"
    assert longest_unique_substring("bbbbb") == 1     # "b"
    assert longest_unique_substring("") == 0
    assert longest_unique_substring("aab") == 2       # "ab"
    print("✓ Longest Unique Substring")

    # 3. Reverse Array in-place
    arr = [1, 2, 3, 4, 5]
    reverse_array(arr)
    assert arr == [5, 4, 3, 2, 1]
    reverse_array(arr, 1, 3)
    assert arr == [5, 2, 3, 4, 1]
    print("✓ Array Reversal")

    # 4. Rotate by Reversal
    arr = [1, 2, 3, 4, 5]
    rotate_array_reversal(arr, 2)
    assert arr == [4, 5, 1, 2, 3]
    rotate_array_reversal(arr, 0)
    assert arr == [4, 5, 1, 2, 3]
    print("✓ Array Rotation via Reversal")

    # 5. Rolling Hash (Rabin-Karp)
    assert rolling_hash_search("hello world", "world") == [6]
    assert rolling_hash_search("aaaaa", "aa") == [0, 1, 2, 3]
    assert rolling_hash_search("abc", "") == []
    print("✓ Rolling Hash (Rabin-Karp)")

    # 6. Minimum Size Subarray Sum
    assert min_subarray_len(7, [2, 3, 1, 2, 4, 3]) == 2  # [4, 3]
    assert min_subarray_len(11, [1, 1, 1]) == 0          # no such subarray
    assert min_subarray_len(4, [1, 4, 4]) == 1           # [4]
    print("✓ Minimum Size Subarray Sum")

    print("\nSliding Window, Swapping & Rolling: all tests passed")

test_sliding_swapping_rolling()


### Discussion & Follow-ups — Sliding Window, Swapping, Rolling

- **"How does the deque approach for sliding window max work?"** → Maintain a deque of indices where values are strictly decreasing. The front is always the max of the current window. When a new element arrives, pop all smaller elements from the back (they can never be the max again). When the window moves, pop the front if it's out of bounds. Each element is pushed and popped at most once → O(N) total. *Beginner tip: Think of it as a "waiting line" where the biggest elements push smaller ones out of the way. The biggest is always at the front.*
- **"What's the rolling property of Rabin-Karp hash?"** → Instead of recomputing the hash of each window from scratch (O(K)), you compute the new hash from the old hash in O(1): remove the outgoing character's contribution, shift, and add the incoming character. This makes string matching O(N+M) on average. *Beginner tip: It's like a moving conveyor belt — when a new item arrives, the old one falls off. You don't re-weigh everything, just adjust for what came and went.*
- **"Why use reversal for array rotation instead of slicing?"** → The slicing approach `arr[d:] + arr[:d]` is O(N) time but O(N) extra space. The three-reversal approach is O(N) time and O(1) extra space — important when memory is constrained or in languages where slicing creates copies. *Beginner tip: Think of reversing a list like flipping a deck of cards — you can achieve any rotation with just a few flips and no extra table space.*
- **"What's the variable window technique for subarray problems?"** → Expand the right pointer, adding elements to the window. When the condition is met (e.g. sum ≥ target), shrink from the left to find the minimum window that still satisfies the condition. This works because the condition is monotonic — if a window works, expanding it also works. O(N) time. *Beginner tip: It's like a telescope — extend it to see more, then retract to find the smallest view that still shows what you need.*

In [ ]:
from collections import OrderedDict

class LRUCache:
    """LRU Cache with O(1) get/put using OrderedDict (Python's hash map + linked list)."""

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        # Move to end (most recently used)
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            # Pop LRU item (first inserted)
            self.cache.popitem(last=False)


In [ ]:
# Manual doubly-linked list implementation (what interviewers actually want to see)

class _Node:
    __slots__ = ("key", "value", "prev", "next")
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None


class LRUCacheManual:
    """LRU Cache with hand-rolled doubly linked list + hash map."""

    def __init__(self, capacity: int):
        self.cap = capacity
        self.map = {}  # key -> Node

        # Dummy head/tail sentinels
        self.head = _Node()
        self.tail = _Node()
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node: _Node) -> None:
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_end(self, node: _Node) -> None:
        node.prev = self.tail.prev
        node.next = self.tail
        self.tail.prev.next = node
        self.tail.prev = node

    def get(self, key: int) -> int:
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._add_to_end(node)
        return node.value

    def put(self, key: int, value: int) -> None:
        if key in self.map:
            node = self.map[key]
            node.value = value
            self._remove(node)
            self._add_to_end(node)
        else:
            if len(self.map) == self.cap:
                # Evict LRU (head.next)
                lru = self.head.next
                self._remove(lru)
                del self.map[lru.key]
            node = _Node(key, value)
            self.map[key] = node
            self._add_to_end(node)


### LRU Cache — Tests & Edge Cases

In [ ]:
def test_lru():
    c = LRUCacheManual(2)
    c.put(1, 1)
    c.put(2, 2)
    assert c.get(1) == 1       # 1 is MRU
    c.put(3, 3)                # evicts 2
    assert c.get(2) == -1      # 2 evicted
    c.put(4, 4)                # evicts 1
    assert c.get(1) == -1
    assert c.get(3) == 3
    assert c.get(4) == 4
    print("LRU: all tests passed")

test_lru()


### LRU — Follow-up: TTL Expiration

Add a `time_to_live` parameter. On `get`, check if the node has expired; if so, remove it and return -1.

In [ ]:
import time

class LRUCacheTTL:
    """LRU with TTL expiration per key."""

    def __init__(self, capacity: int, ttl: float = 5.0):
        self.cap = capacity
        self.ttl = ttl  # seconds
        self.map = {}
        self.head, self.tail = _Node(), _Node()
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node): ...
    def _add_to_end(self, node): ...
    # (same helpers as LRUCacheManual above — omitted for brevity)

    def _is_expired(self, node) -> bool:
        return time.monotonic() - node.timestamp > self.ttl

    def get(self, key: int) -> int:
        if key not in self.map:
            return -1
        node = self.map[key]
        if self._is_expired(node):
            self._remove(node)
            del self.map[key]
            return -1
        self._remove(node)
        self._add_to_end(node)
        return node.value

    def put(self, key: int, value: int) -> None:
        # ... similar with timestamp update
        pass


### LRU — Follow-up: Thread Safety

Use `threading.Lock` (or `RLock` for reentrancy) around all public methods. For reads under CPython GIL a simple lock suffices. For true concurrency, consider `concurrent.futures` or sharding.

```python
import threading

class LRUCacheThreadSafe(LRUCacheManual):
    def __init__(self, capacity: int):
        super().__init__(capacity)
        self._lock = threading.Lock()

    def get(self, key: int) -> int:
        with self._lock:
            return super().get(key)

    def put(self, key: int, value: int) -> None:
        with self._lock:
            super().put(key, value)
```

**Testing race conditions:** Use `threading.Thread` + `barrier` to fire concurrent gets/puts and assert no key is lost and capacity is never exceeded under a stress test.

---
## 2. Top-K Frequent Items (Streaming)

**Why they ask:** Bloomberg systems process high-volume feeds (tickers, news tokens, error codes). Finding top-K is a real primitive.

**What they look for:** O(log K) heap updates; bounded memory; deterministic tie-breaks.

**Follow-ups:**
- Support a sliding window.
- Support deletes.
- Approximate heavy hitters (Count-Min Sketch).

In [ ]:
import heapq
from collections import Counter, defaultdict
from typing import List

class TopKTracker:
    """Maintain top-K frequent items from a stream using a min-heap of size K.

    count: dict[item -> frequency]
    heap:  min-heap of (freq, item) — keeps K largest freqs.
    """

    def __init__(self, k: int):
        self.k = k
        self.count: dict[str, int] = defaultdict(int)
        self.heap: List[tuple[int, str]] = []

    def add(self, item: str) -> None:
        self.count[item] += 1
        freq = self.count[item]

        # If item already in heap, we can't easily update — rebuild or lazy-mark.
        # Simpler approach: rebuild heap from counts every N items, or use
        # a Counter + most_common. But for true streaming we do lazy push.

        # Lazy approach: push new (freq, item) always; heap may have stale entries.
        heapq.heappush(self.heap, (freq, item))

    def top_k(self) -> List[str]:
        # Collect up to K distinct items with highest counts
        seen = set()
        result = []
        # Grab a copy of heap and pull out valid entries
        candidates = []
        while self.heap and len(candidates) < self.k:
            freq, item = heapq.heappop(self.heap)
            # Skip stale entries
            if self.count[item] == freq and item not in seen:
                seen.add(item)
                candidates.append((freq, item))
        # Push back popped entries
        for freq, item in candidates:
            heapq.heappush(self.heap, (freq, item))

        return [item for _, item in sorted(candidates, reverse=True)]


class TopKCounter:
    """Simpler: use Counter.most_common(k) — O(N log K) in batch, fine for moderate streams."""

    def __init__(self, k: int):
        self.k = k
        self.counter: Counter[str] = Counter()

    def add(self, item: str) -> None:
        self.counter[item] += 1

    def top_k(self) -> List[str]:
        return [item for item, _ in self.counter.most_common(self.k)]


### Top-K — Tests & Edge Cases

In [ ]:
def test_top_k():
    tk = TopKCounter(3)
    stream = ["AAPL", "MSFT", "AAPL", "GOOG", "AAPL", "MSFT", "TSLA"]
    for s in stream:
        tk.add(s)
    assert tk.top_k() == ["AAPL", "MSFT", "GOOG"]  # AAPL(3), MSFT(2), GOOG(1)
    print("Top-K: all tests passed")

test_top_k()


### Top-K — Follow-up: Count-Min Sketch (Approximate Heavy Hitters)

For massive streams where exact counting is too expensive, use a **Count-Min Sketch** — a probabilistic data structure with sub-linear space.

**Trade-offs:** Space-efficient, but over-counts due to hash collisions. Tunable via error bound $\epsilon$ and confidence $\delta$.

```python
import hashlib
import struct

class CountMinSketch:
    def __init__(self, epsilon=0.001, delta=0.99):
        width = int(2 / epsilon)
        depth = int(-(delta).bit_length())  # ~ln(1/delta)
        self.depth = max(depth, 4)
        self.width = width
        self.table = [[0] * width for _ in range(self.depth)]

    def _hashes(self, item: str):
        for seed in range(self.depth):
            h = hashlib.sha256((str(seed) + item).encode()).digest()
            yield struct.unpack("<I", h[:4])[0] % self.width

    def add(self, item: str, delta: int = 1):
        for row, col in enumerate(self._hashes(item)):
            self.table[row][col] += delta

    def estimate(self, item: str) -> int:
        return min(self.table[row][col]
                   for row, col in enumerate(self._hashes(item)))
```

**To get top-K with CMS:** Pair it with a heap; the sketch gives approximate frequencies, heap tracks the top-K candidates.

---
## 3. Merge Overlapping Intervals

**Why they ask:** Time windows, trading sessions, and event ranges show up everywhere in financial systems.

**What they look for:** Sort + linear scan, correct boundary handling.

**Follow-ups:**
- Return gaps (complement of merged intervals).
- Process intervals as a stream.
- Compute total covered time.

In [ ]:
from typing import List, Tuple

def merge_intervals(intervals: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Merge overlapping intervals. O(N log N) from sort, O(N) scan."""
    if not intervals:
        return []

    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]

    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:  # overlap (touch counts as overlap for [closed, closed])
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    return merged


### Merge Intervals — Tests & Edge Cases

In [ ]:
def test_merge_intervals():
    assert merge_intervals([]) == []
    assert merge_intervals([(1, 3)]) == [(1, 3)]
    assert merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]) == [(1, 6), (8, 10), (15, 18)]
    assert merge_intervals([(1, 4), (4, 5)]) == [(1, 5)]  # touching
    assert merge_intervals([(6, 8), (1, 9), (2, 4)]) == [(1, 9)]  # full containment
    print("Merge Intervals: all tests passed")

test_merge_intervals()


### Merge Intervals — Follow-ups

**1. Return gaps** — after merging, gaps are the space *between* merged intervals:
```python
def find_gaps(intervals):
    merged = merge_intervals(intervals)
    return [(merged[i][1], merged[i+1][0])
            for i in range(len(merged)-1) if merged[i][1] < merged[i+1][0]]
```

**2. Insert interval and merge** — binary search for insertion point, then merge:
```python
def insert_interval(intervals, new_interval):
    # Insert in sorted order, then run merge_intervals
    ...
```

**3. Total covered time** — sum of `(end - start)` after merging:
```python
def total_covered(intervals):
    return sum(end - start for start, end in merge_intervals(intervals))
```

---
## 4. Currency Conversion (Graph)

**Why they ask:** Bloomberg data is a graph — entities, identifiers, mappings, derivations. Currency conversion is the classic interview example.

**What they look for:** BFS/DFS for unweighted, Dijkstra for weighted, cycle handling.

**Follow-ups:**
- Rates update continuously.
- How do you cache answers?
- How do you bound staleness?

In [ ]:
from collections import defaultdict, deque
import heapq
from typing import Dict, List, Tuple

def build_graph(rates: List[Tuple[str, str, float]]) -> Dict[str, List[Tuple[str, float]]]:
    """Build adjacency list. Edge a->b with rate r means 1 a = r b."""
    graph = defaultdict(list)
    for src, dst, rate in rates:
        graph[src].append((dst, rate))
        graph[dst].append((src, 1.0 / rate))
    return graph


def convert_currency_bfs(rates, src: str, dst: str) -> float:
    """Unweighted: BFS to find any conversion path. Assumes multiplication along edges."""
    graph = build_graph(rates)
    if src not in graph or dst not in graph:
        return -1.0

    visited = {src}
    queue = deque([(src, 1.0)])

    while queue:
        currency, product = queue.popleft()
        if currency == dst:
            return product
        for neighbor, rate in graph[currency]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, product * rate))
    return -1.0


def convert_currency_dijkstra(rates, src: str, dst: str) -> float:
    """Weighted graph (log conversion for multiplicative -> additive)."""
    graph = build_graph(rates)
    if src not in graph or dst not in graph:
        return -1.0

    # Use -log(rate) as edge weight so multiplicative conversion becomes additive
    # Maximizing rate = minimizing -log(rate)
    # dist[currency] = best product so far (initialized to 0)
    dist = defaultdict(float)
    dist[src] = 1.0
    pq = [(-1.0, src)]  # max-heap via negative

    while pq:
        neg_product, currency = heapq.heappop(pq)
        product = -neg_product
        if currency == dst:
            return product
        if product < dist[currency]:
            continue  # stale entry
        for neighbor, rate in graph[currency]:
            new_product = product * rate
            if new_product > dist.get(neighbor, 0):
                dist[neighbor] = new_product
                heapq.heappush(pq, (-new_product, neighbor))
    return -1.0


### Currency Conversion — Tests & Edge Cases

In [ ]:
def test_currency_conversion():
    rates = [
        ("USD", "EUR", 0.92),
        ("EUR", "GBP", 0.86),
        ("USD", "GBP", 0.79),
        ("GBP", "JPY", 190.0),
    ]

    # Direct
    assert abs(convert_currency_bfs(rates, "USD", "EUR") - 0.92) < 1e-6
    # One hop via EUR
    gbp_via_eur = convert_currency_bfs(rates, "USD", "GBP")  # 0.92 * 0.86 = 0.7912
    assert abs(gbp_via_eur - 0.7912) < 1e-3 or abs(gbp_via_eur - 0.79) < 1e-3
    # Unknown currency
    assert convert_currency_bfs(rates, "USD", "XYZ") == -1.0

    # Same currency
    assert convert_currency_bfs(rates, "USD", "USD") == 1.0

    # Dijkstra version
    assert abs(convert_currency_dijkstra(rates, "USD", "EUR") - 0.92) < 1e-6
    assert convert_currency_dijkstra(rates, "USD", "XYZ") == -1.0
    print("Currency Conversion: all tests passed")

test_currency_conversion()


### Currency Conversion — Follow-ups

**1. Rates update continuously** — use a versioned graph. On each rate update, increment a global version counter. Cache entries store the version they were computed at. Before returning a cached result, check if the graph version has changed — if so, invalidate or recompute.

**2. Cache answers** — memoize conversion results with a TTL:
```python
from functools import lru_cache

class CurrencyConverter:
    def __init__(self):
        self._rates = []
        self._graph_version = 0

    @lru_cache(maxsize=1024)
    def convert(self, src: str, dst: str) -> float:
        return convert_currency_bfs(self._rates, src, dst)
```

**3. Bound staleness** — each cached result carries a timestamp. On lookup, if `now - timestamp > max_stale`, recompute. Use a background thread to refresh popular currency pairs before they expire.

---
## 5. Insert / Delete / getRandom — O(1)

**Why they ask:** Tests data structure composition under time pressure. Foundational for Bloomberg's internal data stores.

**What they look for:** Array + hash map + swap-delete trick.

**Follow-ups:**
- Weighted sampling.
- Thread safety.
- Persistence / snapshotting.

In [ ]:
import random
from typing import Any, List

class RandomizedSet:
    """O(1) insert / delete / getRandom using array + hash map + swap-delete."""

    def __init__(self):
        self.arr: List[Any] = []      # values
        self.map: dict[Any, int] = {}  # value -> index in arr

    def insert(self, val: Any) -> bool:
        if val in self.map:
            return False  # already present
        self.map[val] = len(self.arr)
        self.arr.append(val)
        return True

    def delete(self, val: Any) -> bool:
        if val not in self.map:
            return False
        idx = self.map[val]
        last_val = self.arr[-1]

        # Swap with last element
        self.arr[idx] = last_val
        self.map[last_val] = idx

        # Remove last (old val or duplicate)
        self.arr.pop()
        del self.map[val]
        return True

    def get_random(self) -> Any:
        return random.choice(self.arr)


### RandomizedSet — Tests & Edge Cases

In [ ]:
def test_randomized_set():
    rs = RandomizedSet()
    assert rs.insert(1) == True
    assert rs.insert(2) == True
    assert rs.insert(1) == False  # duplicate
    assert rs.delete(2) == True
    assert rs.delete(2) == False  # already gone
    assert rs.get_random() in (1,)  # only 1 remains

    # Stress: insert 100, delete half, ensure all remaining are valid
    rs2 = RandomizedSet()
    for i in range(100):
        rs2.insert(i)
    for i in range(0, 100, 2):  # delete evens
        rs2.delete(i)
    for _ in range(1000):
        val = rs2.get_random()
        assert val % 2 == 1  # only odds remain
    print("RandomizedSet: all tests passed")

test_randomized_set()


### RandomizedSet — Follow-ups

**1. Weighted sampling** — store values with weights in the array; use `random.choices(values, weights=weights)` or binary search on prefix sums of weights.
```python
class WeightedRandomizedSet:
    def __init__(self):
        self.arr = []          # values
        self.weights = []      # parallel weight array
        self.prefix = []       # prefix sums for O(log N) weighted choice
        self.map = {}          # value -> index

    def add(self, val, weight=1):
        # ... maintain prefix sums on update
        pass

    def get_random_weighted(self):
        total = self.prefix[-1]
        r = random.random() * total
        idx = bisect_left(self.prefix, r)
        return self.arr[idx]
```

**2. Thread safety** — wrap with `threading.Lock`. For read-heavy workloads, use `readers-writer` lock or `copy-on-write` for snapshot isolation.

**3. Persistence / snapshotting** — serialize `self.arr` to disk (pickle, JSON, or binary). To snapshot atomically, take a lock, copy the array and map, then serialize the copy. Restore via `__init__` → `__setstate__`.